# 17 — K-Nearest Neighbours (KNN)

**Algorithm:** KNeighborsClassifier (cosine distance, k=5)  
**Features:** TF-IDF (50K) — subsampled to 30K for tractable search  
KNN classifies by majority vote among the k nearest training points. Cosine similarity is the natural distance for TF-IDF vectors. The k vs accuracy plot shows how neighbourhood size affects bias-variance trade-off.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, evaluate_and_print, plot_confusion_matrix,
                   plot_roc_curve, plot_pr_curve)
from sklearn.metrics import accuracy_score
BASE_DIR, FIGURES_DIR = setup_paths()

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2, max_df=0.95)
X_tr_full = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)

# Subsample training set (KNN is O(n) at inference)
KNN_SAMPLE = 30000
np.random.seed(42)
knn_idx = np.random.choice(X_tr_full.shape[0], min(KNN_SAMPLE, X_tr_full.shape[0]), replace=False)
X_tr = X_tr_full[knn_idx]
y_tr = y_train[knn_idx]
print(f'KNN train subset: {X_tr.shape[0]:,}  |  Test: {len(X_test):,}')

## Model Training

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
MODEL_NAME = 'KNN'
model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1, metric='cosine')
t0 = time.time()
model.fit(X_tr, y_tr)
train_time = time.time() - t0
print(f'Fitting completed in {train_time:.2f}s')
metrics, y_pred, y_prob = evaluate_and_print(model, X_te, y_test, MODEL_NAME, train_time)

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## Convergence: k vs Accuracy

In [ ]:
k_values = [1, 3, 5, 7, 10, 15, 20, 30, 50]
tr_accs, te_accs = [], []
for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k, n_jobs=-1, metric='cosine')
    m.fit(X_tr, y_tr)
    tr_accs.append(accuracy_score(y_tr, m.predict(X_tr)))
    te_accs.append(accuracy_score(y_test, m.predict(X_te)))
    print(f'  k={k:>2}: Train={tr_accs[-1]:.4f}  Test={te_accs[-1]:.4f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_values, tr_accs, 'o-', color='#3498db', lw=2, ms=7, label='Train Accuracy')
ax.plot(k_values, te_accs, 's-', color='#e74c3c', lw=2, ms=7, label='Test Accuracy')
ax.axvline(5, color='grey', linestyle='--', lw=1.2, label='k=5 (used)')
ax.set_xlabel('k (Number of Neighbours)')
ax.set_ylabel('Accuracy')
ax.set_title('KNN — k vs Accuracy (Bias-Variance Trade-off)')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'knn_k_vs_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()